# 📈 도킨스의 족제비: 누적 선택 — 데이터 분석 빈칸 실습

원숭이가 타자기를 치면 문장이 나올지 컴퓨터로 흉내 냈어요. 좋은 변이를 남기는 정도는 높으면 목표 문장에 빨리 도달할 수도 있어요. 돌연변이율은 너무 낮거나 높으면 속도가 달라질 수도 있어요. 세대당 자손 수와 목표 문장 길이는 데이터로 확인해 봐요. 문자 종류는 많으면 오래 걸릴 수도 있어요. 세대 시간이나 글자 크기처럼 결과와 상관없을 것 같은 것도 섞여 있어요 — 골라내는 게 숙제! 마구잡이로 치기보다 좋은 것만 남기면 훨씬 빨라요. 진짜로 잰 숫자가 아니라 컴퓨터로 흉내 낸 연습용 데이터예요.

**핵심 속성(변수)**
- **좋은 변이를 남기는 정도(누적 선택)**(%) — 맞는 글자를 남기고 이어 가는 정도예요. 클수록 빨리 목표에 가까워져요.
- **돌연변이율**(확률) — 글자가 우연히 바뀌는 확률이에요. 클수록 뒤죽박죽돼요.
- **세대당 자손 수**(개) — 한 세대에 만드는 문장의 수예요. 클수록 좋은 후보가 많아요.
- **목표 문장 길이**(글자) — 맞혀야 하는 문장의 글자 수예요. 클수록 어려워요.
- **결과** — 목표 문장을 맞히기까지 걸린 세대의 수예요.

**기타 속성** — 결과와 관련이 있을까? 상관을 보고 넣을지 뺄지 **내가 골라요**.
- 문자 종류(가지) — 문장에 쓰는 글자 종류예요.
- 한 세대 시간(초) — 세대가 바뀌는 시간이에요.
- 선택되는 후보 수(개) — 다음 세대로 가는 후보의 수예요.
- 화면 글자 크기(pt) — 화면에 보이는 글자 크기예요.
- 실험 반복 횟수(회) — 같은 실험을 반복한 횟수예요.

> ⚠️ 연습용 합성 데이터(실제 측정 아님)

### 오늘 할 일 (약 30분) — 수업에서 배운 순서 그대로
1. 불러오기 → 2. 결측치 → 3. 이상치(IQR) → 4. 요약 → 5. 그래프 → 6. 상관으로 속성 고르기 → 7. train_test_split → 7-1. 정규화(0~1) → 8. 회귀 모델 → 9. 내 생각

`______` 빈칸에 알맞은 **함수 이름이나 숫자**를 넣고 ▶ 실행해요. 막히면 **오픈북** — 클래스룸에 올린 그간의 실습 파일(Pandas 문법·결측치·이상치·예측·분류)을 펴 보고 같은 코드를 찾아 써요. 단계마다 📖 에 어느 파일 몇 번 숙제인지 적어 두었어요.

In [ ]:
# 0. 준비 — 그대로 실행해요 (한글 그래프 글꼴)
!pip -q install koreanize-matplotlib
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib

## 1. 데이터 불러오기
CSV 파일을 표(DataFrame)로 읽어요. 함수 이름은 **읽다(read) + csv**.

📖 **오픈북** — 「#16 펭귄-분류」 숙제01 `pd.read_csv(url)` · 「Pandas 문법-(A~E)」 `df.head()`

In [ ]:
URL = "https://baehwa-ai-2026.web.app/dataproject/csv/c943_n.csv"
df = pd.______(URL)      # 힌트: CSV 를 읽는 함수
print(df.shape)          # (행 수, 열 수)
df.______()              # 힌트: 앞 5줄만 보기

▶ 이렇게 나오면 맞아요 (숫자는 조금 달라도 괜찮아요)

![](https://raw.githubusercontent.com/SoojiNoh/baehwa-colab/main/nb/img/c943_4_out0.png)
![](https://raw.githubusercontent.com/SoojiNoh/baehwa-colab/main/nb/img/c943_4_out1.png)

## 2. 결측치(빈칸) 처리
빈칸이 몇 개인지 세고, 빈칸이 있는 줄을 지워요. (평균으로 채우는 `fillna(df[열].mean())` 도 배웠죠 — 해 보고 싶으면 바꿔 봐요.)

📖 **오픈북** — 「#16 펭귄-분류」 숙제05 `df.isnull().sum()` · 숙제06 `df.dropna(axis=0)` · 「#12 결측치(seoultemp)」

In [ ]:
print(df.______().sum())   # 힌트: 빈칸인지 True/False — 열마다 개수
df = df.______()            # 힌트: 빈칸 있는 줄 지우기
print('빈칸 지운 뒤', df.shape)

▶ 이렇게 나오면 맞아요 (숫자는 조금 달라도 괜찮아요)

![](https://raw.githubusercontent.com/SoojiNoh/baehwa-colab/main/nb/img/c943_7_out0.png)

## 3. 이상치(IQR) 처리
값을 작은 것부터 줄 세웠을 때 4분의 1 지점이 Q1, 4분의 3 지점이 Q3 예요. 둘 사이 거리를 IQR 이라고 해요(가운데 절반이 퍼진 폭). **Q1 − 1.5×IQR 보다 작거나 Q3 + 1.5×IQR 보다 크면 이상치**로 보고 지워요.

단, 1~4 등급이나 「거의 0인 횟수」처럼 **값 종류가 몇 개뿐인 열은 건너뛰어요** — 그런 열의 드문 값은 이상치가 아니라 정상이에요.

📖 **오픈북** — 「#14 Iris 데이터 전처리(이상치)」 숙제14 `quantile(0.25)` · 숙제15 `Q3 + 1.5 * IQR`

In [ ]:
for col in ["좋은 변이를 남기는 정도(누적 선택)", "돌연변이율", "세대당 자손 수", "목표 문장 길이"]:
    q1 = df[col].______(0.25)        # 힌트: 분위수
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    if iqr == 0 or df[col].nunique() <= 5: continue      # 값 종류가 적은 열은 건너뛰기
    하한 = q1 - 1.5 * iqr
    상한 = q3 + ______ * iqr          # 힌트: 몇 배?
    df = df[(df[col] >= 하한) & (df[col] <= 상한)]
print('이상치 지운 뒤', df.shape)

▶ 이렇게 나오면 맞아요 (숫자는 조금 달라도 괜찮아요)

![](https://raw.githubusercontent.com/SoojiNoh/baehwa-colab/main/nb/img/c943_10_out0.png)

## 4. 요약 통계
평균·최솟값·최댓값을 한 번에 봐요. 이상치를 지운 뒤 값이 그럴듯해졌나요?

📖 **오픈북** — 「#14 Iris 데이터 전처리(이상치)」 숙제03 `df.describe()`

In [ ]:
df.______()               # 힌트: 숫자 열 요약

▶ 이렇게 나오면 맞아요 (숫자는 조금 달라도 괜찮아요)

![](https://raw.githubusercontent.com/SoojiNoh/baehwa-colab/main/nb/img/c943_13_out0.png)

## 5. 관계 그래프
`좋은 변이를 남기는 정도(누적 선택)` 이(가) 바뀔 때 결과는 어떻게 바뀌나요? 점 그래프(산점도)로 봐요.

📖 **오픈북** — 「목둘레 허리둘레-예측」 숙제09 `plt.scatter(X_test, y_test)`

In [ ]:
plt.______(df["좋은 변이를 남기는 정도(누적 선택)"], df["결과(목표 문장에 도달한 세대 수(세대))"])   # 힌트: 점 그래프(산점도)
plt.xlabel("좋은 변이를 남기는 정도(누적 선택)"); plt.ylabel("결과(목표 문장에 도달한 세대 수(세대))")
plt.show()

▶ 이렇게 나오면 맞아요 (숫자는 조금 달라도 괜찮아요)

![](https://raw.githubusercontent.com/SoojiNoh/baehwa-colab/main/nb/img/c943_16_out0.png)

## 6. 상관으로 속성 고르기
결과와의 상관계수예요. 1이나 −1에 가까울수록 관련이 크고, 0 근처면 거의 관련 없어요. **기타 속성은 넣을까요, 뺄까요?**

⚠ 상관이 크다고 「원인」이라는 뜻은 아니에요. 그리고 상관계수는 **직선 관계**만 재요 — 곡선 관계는 0 근처로 나올 수 있어요.

📖 **오픈북** — 「#14 Iris 데이터 전처리(이상치)」 숙제19 `df.corr(numeric_only=True)`

In [ ]:
df.______(numeric_only=True)["결과(목표 문장에 도달한 세대 수(세대))"].sort_values()   # 힌트: 상관계수

▶ 이렇게 나오면 맞아요 (숫자는 조금 달라도 괜찮아요)

![](https://raw.githubusercontent.com/SoojiNoh/baehwa-colab/main/nb/img/c943_19_out0.png)

## 7. 학습용 / 시험용 나누기 — train_test_split
수업에서 배운 그대로 30% 를 시험용으로 떼어 놓아요. X 에는 6번에서 고른 속성을 넣어요.

📖 **오픈북** — 「예측-전복나이」 숙제03 · 「#16 펭귄-분류」 숙제07 `train_test_split(X, y, test_size=0.3, random_state=42)`

In [ ]:
from sklearn.model_selection import train_test_split
X = df[["좋은 변이를 남기는 정도(누적 선택)", "돌연변이율", "세대당 자손 수", "목표 문장 길이"]]   # ← 6번을 보고 속성을 넣거나 빼 봐요
y = df["결과(목표 문장에 도달한 세대 수(세대))"]
X_train, X_test, y_train, y_test = ______(X, y, test_size=0.3, random_state=42)   # 힌트: 나누는 함수
print(len(X_train), len(X_test))

▶ 이렇게 나오면 맞아요 (숫자는 조금 달라도 괜찮아요)

![](https://raw.githubusercontent.com/SoojiNoh/baehwa-colab/main/nb/img/c943_22_out0.png)

## 7-1. 정규화 — 숫자 크기를 0~1 로 맞추기
속성마다 숫자 크기가 달라요(예: 나이 15 와 소득 3,000,000). **가장 작은 값은 0, 가장 큰 값은 1** 이 되게 바꿔요.

기준(최솟값·최댓값)은 **학습용에서만** 구해요 — 시험지를 미리 보면 안 되니까요. 시험용도 같은 기준으로 바꿔요.

선형 회귀는 정규화해도 점수가 거의 같지만, 딥러닝(신경망)은 정규화를 안 하면 잘 못 배워요. 습관처럼 해 둬요.

📖 **오픈북** — 「MNIST 딥러닝」 숙제03 정규화 `X / 255` · 「#20 과일 채소 딥러닝」 min·max 로 0~1

In [ ]:
mn, mx = X_train.min(), X_train.______()      # 힌트: 가장 큰 값 (가장 작은 값은 min)
rng = (mx - mn).replace(0, 1)                 # 값이 한 가지뿐인 열은 0 으로 나누지 않게
X_train = (X_train - mn) / rng
X_test = (X_test - mn) / rng                  # 시험용도 학습용 기준으로
X_train.describe().loc[['min', 'max']]        # 학습용은 0 ~ 1 이 됐나요?

▶ 이렇게 나오면 맞아요 (숫자는 조금 달라도 괜찮아요)

![](https://raw.githubusercontent.com/SoojiNoh/baehwa-colab/main/nb/img/c943_25_out0.png)

## 8. 예측 모델 만들기 (선형 회귀)
학습용으로 맞추고(fit), 시험용으로 점수(score)를 봐요. 1 에 가까울수록 잘 맞아요.

점수가 0 근처거나 **0보다 작으면** 「그냥 평균으로 찍는 것보다도 못하다」는 뜻이에요 — 고른 속성이 결과를 잘 설명하지 못하거나, 관계가 직선이 아닐 수 있어요. 틀린 게 아니라 **발견**이에요. 보고서 「1-5. 한계」에 적어요.

📖 **오픈북** — 「예측-전복나이」 숙제04 `model.fit(X_train, y_train)` · `model.score(X_test, y_test)`

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.______(X_train, y_train)                    # 힌트: 학습시키기
print('시험 점수', model.______(X_test, y_test))   # 힌트: 점수 매기기

▶ 이렇게 나오면 맞아요 (숫자는 조금 달라도 괜찮아요)

![](https://raw.githubusercontent.com/SoojiNoh/baehwa-colab/main/nb/img/c943_28_out0.png)

## 9. (더 해 보기) 분류형 데이터
같은 모형의 분류형 데이터예요. 결과 `결과(오래 걸렸는지 여부)` 별로 속성 평균이 어떻게 다른지 묶어서 봐요.

📖 **오픈북** — 「#14 Iris 데이터 전처리(이상치)」 숙제04 `df.groupby('종류')`

In [ ]:
dc = pd.read_csv("https://baehwa-ai-2026.web.app/dataproject/csv/c943_c.csv").dropna()
# 이상치가 섞이면 평균이 튀어요 — 3번에서 배운 IQR 거르기를 여기서도 해요
for col in ["좋은 변이를 남기는 정도(누적 선택)", "돌연변이율", "세대당 자손 수", "목표 문장 길이"]:
    q1, q3 = dc[col].quantile(0.25), dc[col].quantile(0.75)
    if q3 == q1 or dc[col].nunique() <= 5: continue      # 값 종류가 적은 열은 건너뛰기
    dc = dc[(dc[col] >= q1 - 1.5 * (q3 - q1)) & (dc[col] <= q3 + 1.5 * (q3 - q1))]
dc.______("결과(오래 걸렸는지 여부)").mean(numeric_only=True)   # 힌트: 묶어서(group) 보기

▶ 이렇게 나오면 맞아요 (숫자는 조금 달라도 괜찮아요)

![](https://raw.githubusercontent.com/SoojiNoh/baehwa-colab/main/nb/img/c943_31_out0.png)

## 10. 내 생각 쓰기
아래를 보고서 「1-4. 그래프 설명」·「1-5. 한계」 칸에 옮겨 써요.
- 그래프에서 읽은 숫자 하나와 그 뜻
- 가장 관련이 큰 속성과 그 이유 · 기타 속성을 넣었는지 뺐는지와 그 이유
- 이 결과를 그대로 믿기 어려운 이유 (합성 데이터·지운 줄·빠진 변수 등)